## Set-up

In [ ]:
# Set-up
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import shap
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScaler
from sklearn.metrics import roc_auc_score, precision_score, recall_score, f1_score, roc_curve
from sklearn.model_selection import GroupShuffleSplit, GroupKFold

project_day = pd.read_csv('../datasets/project_day.csv')
project_day['date'] = pd.to_datetime(project_day['date'])
display(project_day.head())
display(project_day.info())
feature_col = [
    'total_steps', 'cv_steps', 'active_hours', 
    'early_morning_steps', 'morning_steps',
    'afternoon_steps', 'evening_steps'
]
K_days = 5 # From patient_stability
MS_target = 1 / 400 # Real-world prevalence
Screen_size = 100000
Sens_target = [0.70, 0.75, 0.80, 0.85, 0.90, 0.95, 0.975, 0.99]


## Screening Burden Analysis

In [ ]:
# Basically start code like patient_stability.ipynb
X = project_day[feature_col].values
y = project_day['label'].values
groups = project_day['id'].values

gss = GroupShuffleSplit(n_splits = 500, test_size = 0.3, random_state = 223) # More splits as we need more stable estimates for granular claims
auc_rows, split_rows = [], []

for split_i, (train_idx, test_idx) in enumerate(gss.split(X, y, groups = groups), start = 1):
        train_data = project_day.iloc[train_idx]
        test_data = project_day.iloc[test_idx]
        
        scaler = RobustScaler()
        X_train_scaled = scaler.fit_transform(train_data[feature_col])
        y_train = train_data['label'].values
        
        healthy = np.sum(y_train == 0)
        ms = np.sum(y_train == 1)
        total = healthy + ms
        class_weight = {
            0: total / (2 * healthy),
            1: total / (2 * ms)
            }
        
        log_mod = LogisticRegression(
            class_weight = class_weight,
            max_iter = 1000,
            random_state = 223
            )
        log_mod.fit(X_train_scaled, y_train)

        train_scores, train_labels = [], []
        for patient_id, g in train_data.sort_values('date').groupby('id'):
             if len(g) >= K_days:
                gK = g.head(K_days)
                X_scaled = scaler.transform(gK[feature_col])
                train_scores.append(log_mod.predict_proba(X_scaled)[:, 1].mean())
                train_labels.append(int(gK['label'].iloc[0]))
        
        test_scores, test_labels = [], []
        for patient_id, g in test_data.sort_values('date').groupby('id'):
             if len(g) >= K_days:
                gK = g.head(K_days)
                X_scaled = scaler.transform(gK[feature_col])
                test_scores.append(log_mod.predict_proba(X_scaled)[:, 1].mean())
                test_labels.append(int(gK['label'].iloc[0]))

        train_scores, train_labels = np.array(train_scores), np.array(train_labels)
        test_scores, test_labels = np.array(test_scores), np.array(test_labels)

        auc_rows.append({
             'split' : split_i,
             'auc' : roc_auc_score(test_labels, test_scores)
        })

        # BEGIN THE SENSITIVITY ANALYSIS PART
        fpr, tpr,thr = roc_curve(train_labels, train_scores, pos_label = 1)
        spec = 1 - fpr

        # Shooting for progressive sensitivity targets to show progression from 0.7-0.99
        for target_sens in Sens_target:
             # Pick point on ROC curve at given sensivity w/ highest specificity
             idx = np.where(tpr >= target_sens)[0]
             if len(idx) == 0:
                  continue
             best = idx[np.argmax(spec[idx])]
             threshold = thr[best]
             pred = (test_scores >= threshold).astype(int)
             tp = np.sum((pred == 1) & (test_labels == 1))
             fp = np.sum((pred == 1) & (test_labels == 0))
             tn = np.sum((pred == 0) & (test_labels == 0))
             fn = np.sum((pred == 0) & (test_labels == 1))

             # Standard sens + spec calculation, 1e-16 for numerical stability, extrapolate to pop of 100k
             sensitivity = tp / (tp + fn + 1e-16)
             specificity = tn / (tn + fp + 1e-16)
             tp_100k = Screen_size / 400 * sensitivity
             fp_100k = Screen_size * (1 - 1 / 400) * (1 - specificity)
             split_rows.append({
                  'split' : split_i,
                  'sensitivity_target' : target_sens,
                  'sensitivity' : sensitivity,
                  'specificity' : specificity,
                  'TP_Per_100k' : tp_100k,
                  'FP_Per_100k' : fp_100k,
                  'Referral_Per_100k' : tp_100k + fp_100k
             })
auc_results = pd.DataFrame(auc_rows)
screen_results = pd.DataFrame(split_rows)

# Aggregate sensitivity, specificity, etc. averages
summary = screen_results.groupby('sensitivity_target', as_index = False).agg(
     mean_sensitivity = ('sensitivity', 'mean'),
     mean_specificity = ('specificity', 'mean'),
     std_specificity = ('specificity', 'std'), 
     mean_TP_100k = ('TP_Per_100k', 'mean'), 
     mean_FP_100k = ('FP_Per_100k', 'mean'),
     mean_Referral_100k = ('Referral_Per_100k', 'mean')
)
summary['mean_PPV'] = summary['mean_TP_100k'] / (summary['mean_FP_100k'] + summary['mean_TP_100k'])
summary['fp_tp_ratio'] = summary['mean_FP_100k'] / summary['mean_TP_100k']

# Generate 4 by 4 Plot
# 1. Specificity at Sensitvity  2. PPV at realistic pop prevalence 3. FP:TP Ratio 4. FP per 100k
fig, axes = plt.subplots(2, 2, figsize = (16, 12))

axes[0, 0].errorbar(summary['sensitivity_target'], summary['mean_specificity'], yerr = summary['std_specificity'], marker = 'o')
axes[0, 0].set(xlabel = 'Sensitivity Target', ylabel = 'Specificity', title = 'Specificity at Sensitivity')

axes[0, 1].plot(summary['sensitivity_target'], summary['mean_PPV'], marker = 'o')
axes[0, 1].set(xlabel = 'Sensitivity Target', ylabel = 'Positive Predictive Value', title = 'PPV at Population Prevalence (1:400)')

axes[1, 0].plot(summary['sensitivity_target'], summary['fp_tp_ratio'], marker = 'o')
axes[1, 0].set(xlabel = 'Sensitivity Target', ylabel = 'False Positives Per True Positive', title = 'FP:TP Ratios')

axes[1, 1].plot(summary['sensitivity_target'], summary['mean_FP_100k'], marker = 'o')
axes[1, 1].set(xlabel = 'Sensitivity Target', ylabel = 'False Positives Per 100k', title = 'False Positives Per 100k at Sensitivity')

# Output and save
plt.tight_layout()
plt.savefig('../output/4by4.png')
plt.show()
display(summary)